# Network Science (981G5) Assessment
This notebook contains the computational pipeline for the Network Science (981G5) assessment. The primary objective of this work is to lay the groundwork for constructing domain-aware, robust null models for benchmark analysis in football analytics.

The notebook is structured into the following sections:

**Sections 1 & 2 (Data Ingestion & Network Construction):** Handles the retrieval and transformation of raw StatsBomb spatiotemporal event logs into directed, weighted passing networks (nx.DiGraph), alongside spatial visualization utilities.

**Sections 3–6 (Diagnostic Metric Showcase):** Demonstrates core network science applications to football, covering Micro, Meso, and Macro metrics (Degree Analysis, Average Shortest Path, Betweenness Centrality, and Transitive Triad Intensity/Clustering).

**Section 7 (Empirical Baselines & The Sparsity Trap):** Illustrates why unconditioned empirical comparisons are fundamentally flawed, establishing the direct necessity for generative reference baselines.

**Section 8 (Failure of Traditional Topological Nulls):** Demonstrates why naive random graph models (Erdős–Rényi and degree-preserving rewiring) fail in spatial sports network modeling.

**Section 9 (Spatially-Constrained Generative Null Engine):** Constructs a 1st-order Spatial Markovian process to synthesize valid reference ensembles:
- **9.1–9.2:** Compiles the season-wide pass corpus and trains the spatial probability tensor.
- **9.3–9.5:** Resamples target match event streams and constructs synthetic null networks.
- **9.6–9.7:** Performs micro/macro degree diagnostics on a single realization, expanding into a 500-iteration Monte Carlo pipeline to log baseline confidence intervals.

**Section 10 (Null-Baselined Tactical Evaluation):** Applies the newly synthesized null ensemble to benchmark the empirical match properties evaluated in Sections 3–6.

---

#### Contents
- [Notebook Setup](#notebook-setup)
- [1. Statbomb API: Raw Data Import](#1-statbomb-api-raw-data-import)
    - [1.1 Competition Extraction](#11-competition-extraction)
    - [1.2 Match, Team, Player and Events Extractions](#12-match-team-player-and-events-extractions)
    - [1.3 League Statistics](#13-league-statistics)
    - [1.4 Highest Match-Team Pass Total](#14-highest-match-team-pass-total)
- [2. Network](#2-network)
    - [2.1 Build Network Function](#21-build-network-function)
    - [2.2 Highest Pass Match Network](#22-highest-pass-match-network)
    - [2.3 Basic Network Visualisation](#23-basic-network-visualisation)
    - [2.4 Pitch Plotting Underlay](#24-pitch-plotting-underlay)
    - [2.5 Plotting Raw Passes](#25-plotting-raw-passes)
    - [2.6 Pitch Plot PassMap](#26-pitch-plot-passmap)
    - [2.7 Frameless Pitch Plot](#27-frameless-pitch-plot)
    - [2.8 Filtered PassMap](#28-filtered-passmap)
    - [2.9 Adjacency Matrix](#29-adjacency-matrix)
- [3. Degree Analysis](#3-degree-analysis)
    - [3.1 Micro-Level Player Metrics](#31-micro-level-player-metrics)
    - [3.2 Macro-Level Network Heterogeneity](#32-macro-level-network-heterogeneity)
        - [Hub Identification](#hub-identification)
- [4. Average Shortest Path](#4-average-shortest-path)
    - [4.1 Global Average Shortest Path Length](#41-global-average-shortest-path-length)
    - [4.2 Player Average Level Path Length](#42-player-average-level-path-length)
    - [4.3 Visualizing Path Efficiency (Node Sizing)](#43-visualizing-path-efficiency-node-sizing)
    - [4.4 Inward vs. Outward Accessibility Scatterplot](#44-inward-vs-outward-accessibility-scatterplot)
- [5. Betweeness Centrality](#5-betweeness-centrality)
    - [5.1 Betweeness Table](#51-betweeness-table)
    - [5.2 Betweeness Node Network Plot](#52-betweeness-node-network-plot)
    - [5.3 Betweeness Bar Chart](#53-betweeness-bar-chart)
- [6. Transitive Triad Intensity (Local Clustering)](#6-transitive-triad-intensity-local-clustering)
    - [6.2 Global Clustering Metrics](#62-global-clustering-metrics)
        - [6.2.1 Mean Global Clustering](#621-mean-global-clustering)
        - [6.2.2 Normalized Global Clustering](#622-normalized-global-clustering)
    - [6.3 Player Level Clustering](#63-player-level-clustering)
    - [6.4 Top Active Transitive Triads](#64-top-active-transitive-triads)
    - [6.5 Overlaying Clustering Triads](#65-overlaying-clustering-triads)
- [7. Empirical Baselines](#7-empirical-baselines)
    - [7.1 Empirical Benchmarking](#71-empirical-benchmarking)
    - [7.2 Distribution of Pass Volumes](#72-distribution-of-pass-volumes)
        - [7.2.1 Histogram](#721-histogram)
        - [7.2.2 Table](#722-table)
        - [7.2.3 Formation Binning](#723-formation-binning)
- [8. Traditional Nulls](#8-traditional-nulls)
    - [8.1 Erdos Renyi (ER) Null Network](#81-erdos-renyi-er-null-network)
        - [8.1.1 Null PassMap Plot](#811-null-passmap-plot)
        - [8.1.2 Null Degree Analysis](#812-null-degree-analysis)
            - [8.1.2.1 Null Player-Level Degree Metrics](#8121-null-player-level-degree-metrics)
            - [8.1.2.2 Null Degree Macro-Level Metrics](#8122-null-degree-macro-level-metrics)
            - [8.1.2.3 Null Top Hubs](#8123-null-top-hubs)
        - [8.1.3 Null Clustering Analysis](#813-null-clustering-analysis)
            - [8.1.3.1 Global Clustering](#8131-global-clustering)
            - [8.1.3.2 Player-Level Clustering](#8132-player-level-clustering)
            - [8.1.3.3 Clustering Triads](#8133-clustering-triads)
            - [8.1.3.4 Clustering Polygon Plot](#8134-clustering-polygon-plot)
    - [8.2 Rewiring Null Approach](#82-rewiring-null-approach)
        - [8.2.1 Rewire vs Network Visual](#821-rewire-vs-network-visual)
        - [8.2.2 Verifying Structural Collapse](#822-verifying-structural-collapse)
- [9. Markovian Null Model](#9-markovian-null-model)
    - [9.1 Construct Pass Dataset and Binning Mechanism](#91-construct-pass-dataset-and-binning-mechanism)
        - [9.1.1 Construct League-Wide Pass Dataset](#911-construct-league-wide-pass-dataset)
        - [9.1.2 Position Counter](#912-position-counter)
            - [9.1.2.1 Original Positions](#9121-original-positions)
            - [9.1.2.2 Condensed Positions](#9122-condensed-positions)
        - [9.1.3 Pitch Grid](#913-pitch-grid)
        - [9.1.4 Plot Pass Bin on Grid](#914-plot-pass-bin-on-grid)
    - [9.2 Recipient Probability Distribution Model](#92-recipient-probability-distribution-model)
        - [9.2.1 Build Spatial Probability Model](#921-build-spatial-probability-model)
            - [9.2.1.1 Query Individual Bin Probability Distributions](#9211-query-individual-bin-probability-distributions)
            - [9.2.1.2 Covert Sample to Player](#9212-covert-sample-to-player)
    - [9.3 Resample Pass Dataset (Recipient)](#93-resample-pass-dataset-recipient)
    - [9.4 Resample Dianostics, Debugging and Sense Checking](#94-resample-dianostics-debugging-and-sense-checking)
        - [9.2.3.1 Percentage of Matching Rewires](#9231-percentage-of-matching-rewires)
        - [9.2.3.2 Change in Match Pass Share](#9232-change-in-match-pass-share)
        - [9.2.3.3 Striker Matrix Bin Probabilties](#9233-striker-matrix-bin-probabilties)
        - [9.2.3.4 Resampled Striker Distribution Receptions](#9234-resampled-striker-distribution-receptions)
        - [9.2.3.5 Sense Check: League Wide Striker Percentages](#9235-sense-check-league-wide-striker-percentages)
    - [9.5 Resample Network](#95-resample-network)
    - [9.6 Degree Analysis](#96-degree-analysis)
    - [9.7 Network Validation](#97-network-validation)
        - [9.2.6.1 Null Validation](#9261-null-validation)
        - [9.2.6.2 Null Summary Statistics](#9262-null-summary-statistics)
- [10. Null Baselined Analysis](#10-null-baselined-analysis)

---

#### Notebook Setup

In [ ]:
# ========================
# Installations
# ========================

# Install required dependencies
%pip install -q --upgrade pip
%pip install -q pandas statsbombpy networkx matplotlib jinja2 seaborn scipy

# Enable live auto-reloading for helpers.py updates
%load_ext autoreload
%autoreload 2

In [ ]:
# ========================
# Imports
# ========================

from typing import Optional, Tuple, Dict, List, Callable
import warnings
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns
from statsbombpy import sb
from statsbombpy.api_client import NoAuthWarning


# Custom helper module
import helper as hp

# Global notebook configurations
warnings.simplefilter("ignore", NoAuthWarning)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

## 1. Statbomb API: Raw Data Import
This the section that works on importing the raw events data from the Statsbomb and processing into a form which is suitable for network analysis. Events data refers to identifable on-field actions, i.e. pass, tackle, shot. 

---

**Section Contents:**
- [1.1 Competition Extraction](#11-competition-extraction)
- [1.2 Match, Team, Player and Events Extractions](#12-match-team-player-and-events-extractions)
- [1.3 League Statistics](#13-league-statistics)
- [1.4 Highest Match-Team Pass Total](#14-highest-match-team-pass-total)

---

### 1.1 Competition Extraction
StatBomb structure their data in various tables and granularity layers. We want to work with the 2023/2024 FA Women's Super League (`competition_id: 37`, `season_id: 281`) data. The code below queries to the `competitions` table to confirm that we have the correct codes. 

In [ ]:
# Retrieve the competitions
competitions_df = sb.competitions()
print(f"Total Competitions Returned: {len(competitions_df)}")

# Filter the DataFrame for WSL 2023/2024
target_filter = (competitions_df['competition_id'] == 37) & (competitions_df['season_id'] == 281)
wsl_competition = competitions_df[target_filter].to_dict(orient='records')[0]

# Print the extracted row
print("\nExtracted Competition Payload:")
print(wsl_competition)

In [ ]:
COMPETITION_ID = 37
SEASON_ID = 281

### 1.2 Match, Team, Player and Events Extractions
To construct a Passing Network (PassMap) we first need to extract and aggregate raw, event-level data across individual matches from the API

1. **Nodes (Players):** Extracted from match lineups to determine player identity, tactical position, and appearance duration.
2. **Edges (Passes):** Extracted from match events to capture successfully completed passes between teammates.
3. **Graph Attributes:** Extracted to store tactical context, such as starting formations.

#### 1.2.1 Extraction Pipeline
The following functions are abstracted to the `helper.py` file:

| Function | Primary Purpose | Output/Pipeline Role |
| :--- | :--- | :--- |
| `hp.fetch_match_details()` | Downloads event stream and lineup payloads in a single API call. | Captures match duration (`max_minute`) and raw payloads. |
| `hp.extract_team_roster()` &<br>`hp.extract_11_players()` | Parses substitution timestamps and calculates individual appearances. | Computes total minutes played and isolates the core 11 players with highest volume. |
| `hp.extract_team_formation()` | Queries tactical setups (e.g., `4-3-3`, `3-4-3`) from starting XI events. | Attaches tactical system metadata to the team network. |
| `hp.extract_successful_passes()` | Filters out incomplete or intercepted passes. | Returns completed pass actions needed to build adjacency matrices. |

The result is a list of **match-team** entries containing: match_id, team name, starting formation, team total passes, the full player roster including subs and the main team of 11 players who played the most minutes. This gives us all of the information be needed to query the API, pull the pass data and construct a network.

In [ ]:
# Extract full seasons worth of match_ids
matches_df = sb.matches(competition_id=COMPETITION_ID, season_id=SEASON_ID)
matches = matches_df["match_id"].to_list()
print(f"WSL 23/24 contains {len(matches)} matches.")

match_records = []

for match in matches:
    events, lineups, max_minute = hp.fetch_match_details(match)
    for team_name, lineup_df in lineups.items():

        # Roster extraction 
        roster = hp.extract_team_roster(lineup_df, max_minute)
        top_11_players = hp.extract_11_players(roster)

        # Tactical and event extraction
        formation = hp.extract_team_formation(events, team_name)
        team_passes = hp.extract_successful_passes(events, team_name)
    
        match_records.append({
            "match_id": match,
            "team": team_name,
            "formation": formation,
            "total_passes": len(team_passes), # Absolute total, not 11
            "full_roster": roster,
            "top_11_players": top_11_players
        })
        
print(f"{len(match_records)} team-match records extracted")
print(40*"=")
print("Sample[0]: ")
print("Match ID:", match_records[0]["match_id"])
print("Team Name:", match_records[0]["team"])
print("Formation:", match_records[0]["formation"])
print("Team Passes:", match_records[0]["total_passes"])
print("Full Roster", len(match_records[0]["full_roster"]))
print("Most Used Players:", len(match_records[0]["top_11_players"]))
print("Most Used Players:", match_records[0]["top_11_players"])
print("Top 3 Players in Roster:")
for p in match_records[0]["top_11_players"][:3]:
    print(
        f"  • {p['Player Name']} ({p['Position']}) - {p['Minutes Played']} mins"
    )

### 1.3 League Statistics
The code pulls some league-level statistics. Of note, we see that there are 105,262 passes accross the whole season, 264 networks (match-team) with an average of 399 passes which ranges from 120 to 847. This is important because passes accumulate to become the edges in our network.

In [ ]:
stats_dict = hp.calculate_league_summary_stats(match_records)

print("\n" + "=" * 60)
print("EXTRACTED SUMMARY STATISTICS (WSL 2023/2024)")
print("=" * 60)
print(f"Total Matches Analyzed:                 {stats_dict['total_matches']}")
print(f"Total Team Match Networks:              {stats_dict['total_team_games']}")
print(f"Total Completed Season Passes:          {stats_dict['total_passes']:,}")
print(f"Mean Passes per Team per Match:         {stats_dict['mean_passes']:.1f} "
f"(Range: {stats_dict['min_passes']} - {stats_dict['max_passes']})")
print(f"Mean Unique Players Used per Game:      {stats_dict['mean_players_used']:.1f} "
f"(Range: {stats_dict['min_players_used']} - {stats_dict['max_players_used']})")
print("=" * 60)

### 1.4 Highest Match-Team Pass Total
The network we will focus on for this project is the match with the highest total of passes recorded by a single team. This match is of note because there is unlikey to be any games for comparison with a similar number of passes. We know that the number of passes impact the network topology and properties of PassMap, therefore, to formally analyse this network we need other networks with similar passes, otherwise the number of passes will dominate the properties. Therefore, this network is a good candidate and demonstrates why we will need null models for analysis.

In [ ]:
# Returns a tuple: (list_index, max_record_dict)
max_idx, highest_pass_match = max(
    enumerate(match_records), 
    key=lambda item: item[1]["total_passes"]
)

print(f"Record Index: {max_idx}")
print(f"Match ID:     {highest_pass_match['match_id']}")
print(f"Team:         {highest_pass_match['team']}")
print(f"Passes:       {highest_pass_match['total_passes']}")

## 2. Network
The Network section outlines the underlying infrastructure and connectivity framework, as well as, visualisation tooling.

---

**Section Contents:**
- [2.1 Build Network Function](#21-build-network-function)
- [2.2 Highest Pass Match Network](#22-highest-pass-match-network)
- [2.3 Basic Network Visualisation](#23-basic-network-visualisation)
- [2.4 Pitch Plotting Underlay](#24-pitch-plotting-underlay)
- [2.5 Plotting Raw Passes](#25-plotting-raw-passes)
- [2.6 Pitch Plot PassMap](#26-pitch-plot-passmap)
- [2.7 Frameless Pitch Plot](#27-frameless-pitch-plot)
- [2.8 Filtered PassMap](#28-filtered-passmap)
- [2.9 Adjacency Matrix](#29-adjacency-matrix)

---

### 2.1 Build Network (PassMap) Function
To transform raw relational event data into formal graph representations, we construct a dedicated pipeline function, `build_passmap_network()`. This pipeline processes and accumulates individual raw event instances from scratch to dynamically construct an attribute-enriched, directed, and weighted graph ($G \in \mathbb{R}^{V \times E}$). The output is a weighted, direct network with node attributes. 
- `extract_top11_pass_events()`: Filters the raw events down to completed passes belonging strictly to the target team and the 11 modeled players, removing subs and injured players.
- `compute_player_average_positions()`: Calculates each player's mean pitch coordinates $(\bar{x}, \bar{y})$ from their pass outward events, binding them to nodes as spatial attributes.
- `aggregate_pass_edges()`: Aggregates pairwise pass counts to assign directed edge weights (e.g., 30 completed passes from Player A to Player B yields an edge weight of 30).


In [ ]:
def build_passmap_network(
        match_record: dict, 
        events_df: Optional[pd.DataFrame] = None,
        null: str=False
        ) -> nx.DiGraph:
    """Constructs a weighted, directed NetworkX graph (nx.DiGraph) for a team passmap.
    
    Parameters
    ----------
    match_record : dict
        A team-match dictionary containing 'match_id', 'team', and 'top_11_players'.
    events_df : pd.DataFrame, optional
        Pre-loaded events DataFrame for the match. If None, fetches directly via API.
            
    Returns
    -------
    nx.DiGraph
        Directed passmap graph.
    """
    # Compile the Match-Team records
    m_id = match_record["match_id"]
    team_name = match_record["team"]
    top_11_players = match_record["top_11_players"]
    top_11_ids = {p['Player ID'] for p in top_11_players}
    player_id_to_name = {p['Player ID']: p['Player Name'] for p in top_11_players}
    
    # API Fallback
    if events_df is None:
        events_df = sb.events(match_id=m_id)

    # Extract raw pass (Edges) and Position (Node) information
    passes_df = hp.extract_top11_pass_events(events_df, team_name, top_11_ids)
    avg_locations = hp.compute_player_average_positions(passes_df)
    
    G = nx.DiGraph(
    team_name=team_name,
    total_passes=match_record["total_passes"]
    )

    # Construct the network nodes from the 11 players
    for p in top_11_players:
        p_id, p_name, p_pos = p['Player ID'], p['Player Name'], p['Position']
        loc = avg_locations.get(p_id, {'x': 50.0, 'y': 50.0})
        
        G.add_node(
            p_name,
            player_id=p_id,
            position=p_pos,
            pos=(loc['x'], loc['y']),
            x=loc['x'],
            y=loc['y']
        )
        
    edges = hp.aggregate_pass_edges(passes_df, player_id_to_name, null=null)
    for passer, recipient, weight in edges:
        G.add_edge(passer, recipient, weight=weight)

    G.graph["true_total_passes"] = sum(d.get('weight', 0) for _, _, d in G.edges(data=True))
        
    return G

### 2.2 Highest Pass Match Network
`G_highest` holds the PassMap network for the highest pass match

In [ ]:
# Pass the dictionary record directly
target_record = match_records[max_idx]
G_highest = build_passmap_network(target_record)

### 2.3 Basic Network Visualisation
Here is an extremely basic visualisation of the network. While the format for the visualisation doesn't matter for most network properties, given we are analysing a spatially constrained game it is useful to see the network well formatted. In this implementation, the nodes are plotted as per their average pass position during the game. This basic implementation doesn't plot the edge weight

In [ ]:
plt.figure(figsize=(4, 3))

# Extract Node Locations
pos = nx.get_node_attributes(G_highest, 'pos')

# Draw basic graph
nx.draw(G_highest, pos=pos)

plt.title("Basic PassMap Network Topology")
plt.axis("on")
plt.grid(True)
plt.show()

### 2.4 Pitch Plotting Underlay Utility
We introduce a custom pitch-underlay function (`draw_vertical_pitch()`) to visualize network nodes over a 2D spatial grid. While purely cosmetic with no effect on underlying graph metrics, rendering nodes at their mean pitch coordinates significantly improves readability and visual appeal. This helper was built natively with Matplotlib to avoid bloated third-party dependencies such as `mlpsoccer`.

In [ ]:
fig, ax = plt.subplots(figsize=(4, 6))

# Draw Pitch Underlay
hp.draw_vertical_pitch(ax=ax)

### 2.5 Plotting Raw Passes
Here I am plotting the raw passes for a match-team and then an individual player within the match. This is demonstate just how messy and unintuative pass data is and why network analysis is so valuable.

In [ ]:

# Create a figure with 1 row and 2 columns
# fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# RAW_TEAM = 'Arsenal WFC'
# RAW_NAME = 'Emily Ann Fox'
# RAW_MATCH = sb.events(match_id=3913160)

# # Plot the individual player passes on the first subplot
# hp.plot_player_raw_passes(RAW_MATCH, RAW_TEAM, RAW_NAME, ax=axes[0])
# axes[0].set_title('Emily Ann Fox')

# # Plot all team passes on the second subplot
# hp.plot_player_raw_passes(RAW_MATCH, RAW_TEAM, None, ax=axes[1])
# axes[1].set_title('Arsenal WFC (All Passes)')

# plt.tight_layout()
# plt.show()

### 2.6 PassMap Network Pitch Plot
This section plots the network overlayed on the pitch. Typically, this is the most intuitive way to plot PassMaps as it gives us insight as to specific areas of the pitch that players operate in and provides us insight as to why certain relationships, clusters and pass routes may be forming. However, this network is very dense with pass volume and as a result makes it difficult to observe the edges. 

In [ ]:
# 1. Unpack fig and ax explicitly
fig, ax = plt.subplots(figsize=(6.5, 9.0), facecolor='#ffffff')

# 2. Call the passmap function with the explicitly assigned ax
hp.plot_passmap_on_pitch(G_highest, ax=ax)

plt.show()

### 2.7 PassMap Network Frameless Plot
This visual retains the spatial coordinates but maximise the space in the images framing. 

In [ ]:
# Declare the figure window
# fig, ax = plt.subplots(figsize=(6.5, 9.0), facecolor='#ffffff')

# # Plot frameless graph
# hp.plot_passmap_frameless(G_highest, ax=ax)

# # Remove all external margins
# plt.tight_layout(pad=0.5)
# plt.show()

### 2.8 Filtered PassMap
As the PassMap represents cumulative passes across an entire match it can be good to set a threshold from which edges excluded from the plot. This is partly done to improve the view of the network by "removing" edges. But functionally, this removes edges which don't represent any sort of meaningful relationship. Two players on the opposite sides of the pitch may make a single interaction in a defensive situation whereby their true positional roles have been compromised. Doing this intensifies stronger links. Note, this is generally just a cosmetic action, we don't tend to actually want to remove edges from the network itself.

In [ ]:
# Filter graph to only show pass combinations made at least 8 times
# G_threshold_8 = hp.filter_graph_edges(G_highest, min_weight=5)

# # Declare the figure window
# fig, ax = plt.subplots(figsize=(6.5, 9.0), facecolor='#ffffff')
# # hp.plot_passmap_on_pitch(G=G_threshold_8, ax=ax)
# hp.plot_passmap_frameless(G_threshold_8, ax=ax)
# plt.tight_layout(pad=0.5)
# plt.show()

### 2.9 Adjacency Matrix

In [ ]:
import networkx as nx
import pandas as pd

# Convert to a labeled Pandas DataFrame
adj_df = nx.to_pandas_adjacency(G_highest, weight='weight', dtype=int)

print(adj_df)

adj_matrix = nx.to_numpy_array(G_highest, weight='weight')

print(adj_matrix)

## 3. Degree Analysis
Analysing networks using degree-based metrics provides a computationally efficent and intuative direct insight to the network.  

---

**Section Contents:**
- [3.1 Degree Analysis Helper Function](#31-degree-analysis-helper-function)
- [3.2 Micro-Level Metrics](#32-micro-level-metrics)
    - [3.2.1 Hub Identification](#321-hub-identification)
- [3.3 Macro-Level Metrics](#33-macro-level-metrics)

---

### 3.1 Degree Analysis Helper Function
To evaluate the structural balance and individual workloads within the passing network, we implement the `analyze_degree_and_heterogeneity()` pipeline. This function extracts both micro-level (player) and macro-level (team) **degree** network properties from the directed pass graph ($G$).

In [ ]:
player_df, macro_stats, top_hubs_df = hp.analyze_degree_and_heterogeneity(G_highest, top_n_hubs=5)

### 3.2 Micro-Level Metrics
Micro-level degree analysis evaluates the individual node (or player, in a football network) rather than the global topology of the entire graph. Focusing on micro-level degree metrics allows us to profile and categorize nodes in the network. In football terms we map this to tactic roles.


| Metric | Notation | Explanation |
| :--- | :--- | :--- |
| Unweighted in/out-degree | $k_{in}, k_{out}$ | Measures the diversity of passing partners. |
| Weighted in/out-strength | $s_{in}, s_{out}$ | Quantifies total passes received and completed. |
| Net Flow | $\Delta s_i = s_{out} - s_{in}$ | Measures directional asymmetry to identify net distributors ($\Delta s_i > 0$) versus net receivers ($\Delta s_i < 0$). |
| Pass Ratio | $s_{out} / s_{in}$ | Identifies positional roles, distinguishing build-up playmakers ($> 1.0$) from target finishing endpoints. |

In [ ]:
print("\n" + "="*90)
print("####Micro-Level Execution: Player-Level Degree Metrics (Arsenal WFC)")
print("="*90)
print(player_df.to_string())

#### 3.2.1 Hub Identification
Ranks the top $N$ players by total pass volume ($s_{tot}$) and calculates their **Relative Volume** ($s_i / \langle s_{tot} \rangle$) to highlight the team's primary playmakers relative to the squad average.

In [ ]:
print("\n" + "="*60)
print("Macro-Level Metrics: Centralization & Network Heterogeneity")
print("="*60)
for metric_name, val in macro_stats.items():
    print(f"{metric_name:<45}: {val:.4f}")

### 3.3 Macro-Level Metrics
Macro-level degree analysis evaluates the global topological properties and overall distribution of connections across an entire network, rather than focusing on individual nodes. 

| Metric | Notation | Formula / Code Variable | What It Measures |
| :--- | :--- | :--- | :--- |
| **Mean Unweighted Degree** | $\langle k \rangle$ | `mean_k` | Measures the average number of unique passing connections per player across the team. |
| **Degree Variance** | $\text{Var}(k)$ | `var_k` | Captures the spread of unique passing connections around the team mean. |
| **Degree Standard Deviation** | $\sigma_k$ | `std_k` | Quantifies the average dispersion of player connection counts from the squad average. |
| **Second Moment** | $\langle k^2 \rangle$ | `second_moment` | Measures the second moment of the unweighted degree distribution to emphasize the presence of highly connected hubs. |
| **Coefficient of Variation** | $CV_k$ | `cv_k` ($\sigma_k / \langle k \rangle$) | Measures unweighted degree heterogeneity to assess how unevenly unique passing channels are distributed across all players. |

In [ ]:
# 3. Top Network Hubs
print("\n" + "="*60)
print("Top Network Hubs (Rank-Ordered)")
print("="*60)
print(top_hubs_df.to_string(index=False))

## 4. Average Shortest Path
Analysing the Average Shortest Path (ASP)—the average number of steps it takes to travel between any two nodes in a network—provides key insights into how efficiently information, resources, or signals flow through a system. ASP uncover whether a network is small-world or not.

---

**Section Contents:**
- [4.1 Average Shortest Path Helper Function](#41-average-shortest-path-helper-function)
- [4.2 Global Average Shortest Path Length](#42-global-average-shortest-path-length)
- [4.3 Player Average Level Path Length](#43-player-average-level-path-length)
- [4.4 Visualizing Path Efficiency (Node Sizing)](#44-visualizing-path-efficiency-node-sizing)
- [4.5 In vs. Out Accessibility Scatter](#45-in-vs-out-accessibility-scatter)

---

### 4.1 Average Shortest Path Helper Function
To evaluate how efficiently the ball circulates across the team, we calculate the all-pairs shortest path length using Dijkstra's algorithm, `all_pairs_dijkstra_path_length()`. Because passing networks represent volume rather than physical distance, edge weights are inverted ($l_{ij} = 1 / w_{ij}$) so that high-frequency passing channels translate into short communication distances.

In [ ]:
d_global_arsenal, dist_matrix_df, player_path_df = hp.calculate_average_shortest_path(G_highest)

### 4.2 Global Average Shortest Path Length
The global average shortest path length ($d$) measures the overall structural efficiency and compactness of the team's passing network.

In [ ]:
print("\n" + "="*80)
print(f"Part 1: Global Team Circulation Distance (d) - Arsenal WFC")
print("="*80)
print(f"Global Average Shortest Path Length (d): {d_global_arsenal:.4f} topological units")

### 4.3 Player Average Level Path Length
To understand individual roles within the passing network, we disaggregate path lengths into inward accessibility ($d_{in}$) and outward reachability ($d_{out}$):

**Mean Outward Path Length ($d_{out}$):** Measures how efficiently a player can start a sequence and reach all other teammates through the passing network. Low $d_{out}$ values identify central build-up distributors who anchor possession. 

**Mean Inward Path Length ($d_{in}$):** Measures how easily all other teammates can route the ball into a specific player. This is critical for players positioned higher up the pitch or specialized targets who require longer or more deliberate sequences to access.

In [ ]:
print("\n" + "="*80)
print("Player-Level Path Length Summary (Ranked by Outward Efficiency)")
print("="*80)
print(player_path_df.to_string())

### 4.4 Visualizing Path Efficiency (Node Sizing)
We can embed these shortest path metrics directly into the spatial PassMap visualization. The `plot_passmap_on_pitch()` function accepts a node_scores argument, allowing us to dynamically scale player node sizes based on path efficiency (where lower path lengths indicate higher efficiency and are mapped to larger node sizes).

In [ ]:
# Extract outward path lengths (d_out) into a dictionary
d_out_dict = player_path_df["Mean Outward Path Length (d_out)"].to_dict()

# Invert scores for node sizing (Smaller d_out = More efficient = LARGER node)
max_path = max(d_out_dict.values()) if d_out_dict else 1.0
path_efficiency_scores = {player: (max_path - score) for player, score in d_out_dict.items()}

# Plot passmap using your updated plot_passmap_on_pitch function
hp.plot_passmap_on_pitch(
    G=G_highest,
    node_scores=path_efficiency_scores,
    metric_name="Mean Shortest Path (d_global)",
    global_metric=f"{d_global_arsenal:.3f}"
)

### 4.5 In vs. Out Accessibility Scatter
Beyond spatial pitch visualisations, we can evaluate player roles by plotting a scatter distribution comparing each player's **Mean Inward Path Length ($d_{in}$)** against their **Mean Outward Path Length ($d_{out}$)**.

**Central Connectors & Build-up Hubs:** Players like Lotte Wubben-Moy and Kim Little cluster tightly at the lower end of both inward and outward paths, showing they are easily found by teammates and efficiently distribute the ball outward

**Peripheral Outlets & Outliers:** Stina Blackstenius (Center Forward) stands out as a clear outlier with significantly higher path lengths ($d_{out} \approx 0.566$, $d_{in} \approx 0.339$). She operates as the terminal forward meaning opposing defensive structures intentionally restrict passing lanes into her, resulting in higher topological distances from the rest of the build-up network. Similarly, attacking midfielders like Bethany Mead exhibit higher path lengths due to their advanced pitch positions, where casual possession loops are harder to establish.  


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# ==============================================================================
# EXECUTION
# ==============================================================================
hp.plot_path_length_scatterplot(
    player_path_df=player_path_df,
    d_global=d_global_arsenal,
    team_name=G_highest.graph.get('team_name', 'Team')
)

## 5.Betweeness Centrality
To move beyond simple passing volume and identify which players act as essential routing bridges within the team's tactical architecture, we evaluate **Betweenness Centrality ($g(i)$)**. Betweenness centrality measures the proportion of all shortest paths passing through a given player, highlighting who controls the flow of possession between disparate zones of the team. 

---

**Section Contents:**
- [5.1 Betweeness Helper Function](#51-betweeness-helper-function)
- [5.2 Betweeness Table](#52-betweeness-table)
- [5.3 Betweeness Node Plot](#53-betweeness-node-plot)
- [5.4 Betweeness Bar Chart](#54-betweeness-bar-chart)

---

### 5.1 Betweeness Helper Function
This computes weighted shortest-path betweenness centrality for our directed passing network. NetworkX evaluates all-pairs shortest paths over these distances and computes the fraction of times each player sits directly on the shortest communication route between any other two teammates. Just like our shortest path analysis, the function converts raw pass counts into distance metrics ($1.0 / \text{weight}$), ensuring higher pass volumes represent shorter communication paths.

In [ ]:
betweenness_df, betweenness_dict = hp.calculate_betweenness_centrality(G_highest)

### 5.2 Betweeness Table
The most simple way to analyse betweenness is to just plot a sorted table. Larger values are more central and important to the bridging of pass routes. Following the liturature, the CBs are the highest scoring players. Not because they directly faciliate a bridge from back to front but because they are constant availble passing option in cyclical possesion.

In [ ]:
print("\n" + "="*80)
print("Part 2: Betweenness Centrality Analysis - Arsenal WFC")
print("="*80)
print(betweenness_df.to_string())

### 5.3 Betweeness Node Plot
Given Betweenness is a player metric, we can visualise it by varying the size of the players nodes. Doing this allows the "hubs" to start out and identify any assymetic in flanks. 

In [ ]:
hp.plot_passmap_on_pitch(
    G=G_highest,
    node_scores=betweenness_dict,
    metric_name="Betweeness Centrality",
)

### 5.4 Betweeness Bar Chart
Here is another example of how network properties can be visualised abstract from networks themselves. It is much easier for us to compare and see the difference between the higher scoring nodes this way. 

In [ ]:
bet_plot = betweenness_df.reset_index()
bet_players = bet_plot["Player"]
bet_scores = bet_plot["Betweenness Centrality g(i)"]

# Plot horizontal bar chart
plt.figure(figsize=(8, 5))
plt.barh(bet_players, bet_scores, color="crimson", edgecolor="black")

# Format chart, invert y-axis so highest score is at the top
plt.gca().invert_yaxis()
plt.xlabel("Betweenness Centrality Score $g(i)$", fontweight="bold")
plt.title("Betweenness Centrality Ranking — Arsenal WFC", fontweight="bold")
plt.grid(axis="x", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

## 6. Clustering (Transitive Triads)
While degree and betweenness centralities capture individual reach and routing control, they treat connections in isolation. To measure local clustering and cohesive group dynamics within a passing network, we evaluate Pure Transitive Triad Intensity ($I_{transitive}$).

In football passing networks, standard unweighted clustering coefficients can be misleading because they treat all connections equally. Transitive triads specifically isolate progressive wall-passes and multi-option triangle combinations ($A \to B$, $B \to C$, $A \to C$). Essentially, this maps relationships focused on advancing or transitioning the ball from A to C with the support of B. Note, we avoid measuring complete, closed-loop triangles ($A \to B \to C \to A$), as cyclic passing rarely indicates tactical progression in football. Because our network is weighted by pass volume, we weight each triad by its weakest link ($W_{min} = \min(W_{AB}, W_{BC}, W_{AC})$). If the $A \to C$ connection is strong but $A \to B$ is weak, it does not constitute a true functional cluster; the direct relationship between A and C simply exists independently.

---

**Section Contents:**
- [6.1 Clustering Helper Function](#61-clustering-helper-function)
- [6.2 Global Clustering Metrics](#62-global-clustering-metrics)
- [6.3 Player Level Clustering](#63-player-level-clustering)
- [6.4 Top Active Transitive Triads](#64-top-active-transitive-triads)
- [6.5 Overlaying Clustering Triads](#65-overlaying-clustering-triads)

---

### 6.1 Clustering Helper Function

In [ ]:
global_internal_clust, global_norm_clust, player_clust, triad_clust = \
    hp.calculate_transitive_triad_intensity(G_highest)

### 6.2 Global Clustering Metrics
In our implementation, each player accumulates a score based on the strength and volume of the transitive triads in which they participate. Players involved in numerous high-volume triads earn higher individual scores. From these individual values, we compute two global clustering metrics:

##### 6.2.1 Mean Global Clustering
This metric calculates the simple mean of all individual player clustering scores across the team:

$$\bar{I}_{transitive} = \frac{1}{N} \sum_{i=1}^{N} I_{transitive}(i)$$

It serves as an internal benchmark to evaluate relative player contribution. Players scoring above the mean act as key linkage hubs who sustain local combinations. However, this metric is less suitable for cross-team comparisons because teams with higher total pass volumes naturally generate higher raw triad scores. Consequently, it cannot distinguish whether a team is inherently more tactically cohesive or simply completing more total passes. A high(er) value of 582.55 might indicidate the strong presence of short-passing combinations and positional triangles rather than direct, long passes or individual dribbling. Though we can't say for sure. 


In [ ]:
print("\n" + "="*80)
print(f"Global Team Transitive Triad Clustering (Mean): {global_internal_clust:.4f} pass units")
print("\n" + "="*80)

##### 6.2.2 Normalized Global Clustering
To enable valid cross-team analysis, this approach normalizes the mean clustering score against the team's internal range of scores (or total pass volume):
$$I_{norm} = \frac{\bar{I}_{transitive} - I_{min}}{I_{max} - I_{min}}$$
This informs us how evenly triad participation is distributed across the entire squad rather than relying on a few isolated key players. Comparing this normalized score across different teams reveals true structural cohesion, allowing us to identify whether a team's build-up play relies on team-wide fluid combinations or isolated, localized interactions.


In [ ]:
print("\n" + "="*80)
print(f"Global Team Transitive Triad Clustering (Norm): {global_norm_clust:.4f}")
print("\n" + "="*80)

### 6.3 Player Level Clustering
This highlights the Double-Pivot Engine of Kim Little ($1.000$) and Victoria Pelova ($0.854$) who dominate in the triad metrics. Many triangular passing combinations flow through the central double-pivot meaning they act as the primary link between the defense and attack. 

The central defensive players Wubben-Moy ($0.790$) and Leah Williamson ($0.738$) also record high triad values showing that Arsenal heavily utilizes short, triangular combinations out of the back to bypass opponent pressing lines rather than long, direct clearances

Steph Catley ($0.565$) records noticeably higher triad involvement than Emily Fox ($0.430$). Combined with Wubben-Moy's edge over Williamson, this indicates a strong tactical preference for building up along the left channel.

Alessia Russo shows as a Hybrid Link operating in a deeper central attacking role here, Alessia Russo ($0.561$) posts strong triad involvement, showing she frequently drops deep to connect with Little and Pelova. Note, that she does not show up in Shortest Path. This may be due to her linking up with general possession, i.e. not the shortest, or should this metrics abiltiy to capture all players in a cluster, with her being often the target player C.

Also, note that Stina Blackstenius is still 0 for this. She doesn't follow Russo's inclusion parameters, she is simply the final point always and not involved in possession.

In [ ]:
print("\n" + "="*80)
print("Player-Level Transitive Triad Summary")
print("="*80)
print(player_clust.to_string())

### 6.4 Top Active Transitive Triads
By extracting the active triads ranked by bottleneck capacity, we can isolate the exact tactical triangles where the team successfully executed passing loops.

In [ ]:
print("\n" + "="*80)
print("Top 10 Active Transitive Triads")
print("="*80)
top_triads_summary = []
for idx, t in enumerate(triad_clust[:10], start=1):
    p1, p2, p3 = t["players"]
    top_triads_summary.append({
        "Rank": idx,
        "Player 1 (Origin/Target)": p1,
        "Player 2 (Intermediate)": p2,
        "Player 3 (Target/Origin)": p3,
        "Total Capacity (Pass Units)": round(t["total_capacity"], 2)
    })

top_triads_df = pd.DataFrame(top_triads_summary).set_index("Rank")
print(top_triads_df.to_string())

### 6.5 Overlaying Clustering Triads
A good way to visualise the key triads is by overlaying the triad clusters on the PassMap. This can help to provide structure to the otherwise chaotic PassMap, allowing us to transform edge density into coordinate passing clusters

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 9.0), facecolor='#ffffff')

hp.draw_vertical_pitch(ax=ax)
hp.plot_passmap_on_pitch(G_highest,ax=ax)
hp.plot_transitive_triads_updated(G=G_highest, triads=triad_clust, top_n=4, ax=ax)

title=G_highest.graph.get('team_name', 'Team')
ax.set_title(f"Top Transitive Triads (Passing Triangles)\n{title}", )
plt.tight_layout()
plt.show()

## 7. Empirical Baselines
Some network properties allow us to infer information about the network abstract from any baseline, i.e. if something has hetrogenity. However, for the purpose of football, most of our analysis will need tobe benchmarked or reference. For anything, the obvious starting point for comparisons is other literal networks, empirical comparisons.

---

**Section Contents:**
- [7.1 League Level Shortest Path Helper Function](#71-league-level-shortest-path-helper-function)
- [7.2 Empirical Benchmarking](#72-empirical-benchmarking)
- [7.3 Distribution of Pass Volumes](#73-distribution-of-pass-volumes)
    - [7.3.1 Histogram](#731-histogram)
    - [7.3.2 Table](#732-table)
    - [7.3.3 Formation Binning](#733-formation-binning)

---

### 7.1 League Level Shortest Path Helper Function
`compute_league_global_shortest_paths()` worss through the `match_records` and compiles a network for each match-team instance and then computes its global shortest path. Resulting, we have a range of metric values to try and benchmark our network against

In [ ]:
league_sp = hp.compute_league_global_shortest_paths(match_records)
league_sp.sort_values(by="d_global", ascending=True, inplace=True)
league_sp

### 7.2 Empirical Benchmarking
Empirical benchmarking clearly ranks our network ($0.1884$) as one of the most efficent networks in the league. It ranks at the top 10 network, $3.41$% and very close to the league minimum of $0.1748$. 

In [ ]:
rank, percentile, series_stats = hp.evaluate_global_d(G_highest, league_sp)

### 7.3 Distribution of Pass Volumes
Given we selected the match with the highest overall pass volume, our network represents a top-percentile (>99th percentile) outlier. Even when grouping matches into broad 100-pass bins, the volume distribution remains extremely sparse at higher ranges. This sparsity poses a critical methodological challenge. Network properties are strongly shaped by node degrees (pass counts); reliable empirical benchmarking requires comparing matches with similar pass volumes. Finding comparable matches is impossible for our selected game and difficult for most matches in the dataset. Finally, empirical baselines require an even distribution of distinct teams within each bucket. Otherwise, benchmarks risk comparing teams primarily against themselves. Together, these factors highlight the severe data-sparsity constraints inherent to constructing empirical baselines.


#### 7.3.1 Histogram

In [ ]:
hp.plot_league_pass_distribution(league=league_sp, G=G_highest)

#### 7.3.2 Table
The 700–799 pass range contains only our selected match. Neighboring ranges are similarly sparse: 600–699 passes contains 8 matches, 500–599 contains 14, and 100–199 contains 22. Even the most populated range (200–299 passes) accounts for only 94 of the 234 total matches ($40.2\%$).

In [ ]:
bin_edges = list(range(0, 1001, 100))

bin_counts = pd.cut(league_sp["Total_Passes"], bins=bin_edges, right=False
    ).value_counts(sort=False)

bin_table = pd.DataFrame({
        "Pass Range": [f"{int(b.left)}–{int(b.right) - 1}" for b in bin_counts.index],
        "Match Count (n)": bin_counts.values,
        "Percentage (%)": (bin_counts.values / len(league_sp) * 100).round(2),
    })

bin_table

#### 7.3.3 Formation Binning
Adding tactical variables further exacerbates this issue. To ensure fair comparisons, matches must also be split by tactical formation, which dictates baseline network topology. As Section 7.2.3 shows, segmenting the largest pass-volume bin (94 matches) across 11 formations leaves the most common setup (4-2-3-1) with only 29 matches ($30.9\%$).

In [ ]:
# Filter bin and get counts + percentages
bin_df = league_sp[league_sp["Total_Passes"].between(300, 399)]
formation_counts = bin_df["Formation"].value_counts()

formation_summary = pd.DataFrame({
    "Match Count (n)": formation_counts,
    "Percentage (%)": (formation_counts / len(bin_df) * 100).round(2)
}).reset_index(names="Starting Formation")

print(f"Total Matches: {len(bin_df)} | Unique Formations: {len(formation_summary)}")
print(formation_summary)

## 8. Traditional Nulls
Traditional null approaches face significant hurdles when applied to association football. Generating a network null model involves randomizing certain topological features while preserving selected properties, such as node degree. Football is inherently spatially constrained by physics and the dimensions of the pitch. Furthermore, tactical and behavioral nuances dictate network topology. Traditional null models fail to account for these domain-specific constraints, often yielding nonsensical graph structures—such as unnatural cross-field connections or high-degree hubs in impossible positions.

---

**Section Contents:**
- [8.1 Erdos Renyi (ER) Null Network]()
    - [8.1.1 ER Helper Function]()
    - [8.1.2 Null PassMap Plot]()
    - [8.1.3 Null Degree Analysis]()
        - [8.1.3.1 Null Player-Level Degree Metrics]()
        - [8.1.3.2 Null Degree Macro-Level Metrics]()
        - [8.1.3.3 Null Top Hubs]()
    - [8.1.4 Null Clustering Analysis]()
        - [8.1.4.1 Global Clustering]()
        - [8.1.4.2 Player-Level Clustering]()
        - [8.1.4.3 Clustering Triads]()
        - [8.1.4.4 Clustering Polygon Plot]()

---

### 8.1 Erdos Renyi (ER) Null Network
The simplest approach to constructing a network null model is the Erdős–Rényi (ER) random graph model. 

#### 8.1.1 ER Helper Function
Under the $G(N, p)$ formulation, an ER network is generated by taking $N$ nodes and drawing an edge between every pair of nodes independently with a uniform probability $p$. In the context of weighted networks, this approach preserves the total weight (e.g., total passes) across the network, but completely destroys the underlying topological structure. The network is rebuilt by distributing edge weights incrementally, one unit at a time, based on a probability distribution derived from the total network weight and node count. Because edge placement is entirely random, all nodes are treated as structural equals, resulting in a homogeneous degree distribution where individual node degree constraints are discarded.

In [ ]:
# Null Construction
G_null_er = hp.generate_erdos_renyi_null_incremental(G_highest, seed=432)

# Verify invariants
print(f"Nodes Preserved: {len(G_null_er.nodes())}")
print(f"Note Attributes: {G_null_er.nodes[list(G_null_er.nodes())[0]]}")
print(f"Total Passes:    {sum(d['weight'] for _, _, d in G_null_er.edges(data=True))}")

# Compute betweenness scores for Null
betweenness_df_ER, betweenness_dict_ER = hp.calculate_betweenness_centrality(G_null_er)


#### 8.1.2 Null PassMap Plot
Plotting the expected goals (ER) PassMap immediately reveals structural anomalies. The network resembles a dense "hairball," with unilateral connections spanning the entire length of the pitch and no noticeable variance in edge widths. Additionally, the striker unexpectedly appears as the central otuward passing hub — an outcome that, while technically possible, is highly improbable to this degree. Similarly, the goalkeeper exhibits consistent, cross-field links with multiple outfield players. When applying a 10-pass minimum filter, the original network dropped many connections, but this plot retains nearly all of them—including those involving the goalkeeper.

In [ ]:
# Filter graph to only show pass combinations made at least 8 times
G_null_thres = hp.filter_graph_edges(G_null_er, min_weight=10)

FIG_SIZE=(6.5, 9.0) # FIG_SIZE=(18, 27)
fig, ax = plt.subplots(figsize=FIG_SIZE, facecolor='#ffffff')
hp.plot_passmap_frameless(
    G=G_null_thres, 
    ax=ax,
    node_scores=betweenness_dict_ER,
);

#### 8.1.3 Null Degree Analysis

In [ ]:
# Analyze graph using the integrated function
player_df_er, macro_stats_er, top_hubs_df_Er = hp.analyze_degree_and_heterogeneity(G_null_er, top_n_hubs=5)

##### 8.1.3.1 Null Player-Level Degree Metrics

In [ ]:
# 1. Player-Level Table
print("\n" + "="*90)
print("#### 4.1. Micro-Level Execution: Player-Level Degree Metrics (Arsenal WFC)")
print("="*90)
print(player_df_er.to_string())

##### 8.1.3.2 Null Degree Macro-Level Metrics
The macro level degree analysis highlights that the ER null network no longer looks like a football network. **Second Moment** ($\frac{\langle k^2 \rangle}{\langle k \rangle} = 15.7176$) is nearly identidical to to the mean degree ($\langle k \rangle = 15.4545$ confirming the absence of playmaking hubs or heavy-tailed degree distributions. **Coefficient of Variation** ($CV_k = 0.1305$) and narrow Degree Variance ($\text{Var}(k) = 4.0661$) demonstrate that random edge generation flattens the network. The low **Node Volume Variance** ($\text{Var}(s_{\text{tot}}) = 514.9917$) (squared) shows that uniform edge-weight distribution erases true tactical workload inequality, scattering total pass volume evenly across all players.

$$\sigma_{s_{\text{tot}}} = \sqrt{\text{Var}(s_{\text{tot}})} = \sqrt{514.9917} \approx \mathbf{22.69 \text{ passes}}$$

In [ ]:
# 2. Macro Network Metrics
print("\n" + "="*60)
print("Macro-Level Metrics: Centralization & Network Heterogeneity")
print("="*60)
for metric_name, val in macro_stats_er.items():
    print(f"{metric_name:<45}: {val:.4f}")

##### 8.1.3.3 Null Top Hubs

In [ ]:
# 3. Top Network Hubs
print("\n" + "="*60)
print("Top Network Hubs (Rank-Ordered)")
print("="*60)
print(top_hubs_df_Er.to_string(index=False))

#### 8.1.4 Null Clustering Analysis
An intuitive way to present the disfigurement of the null network is to compute the clustering triads. Clustering is a downstream metric that imputes several moving parts.

In [ ]:
global_internal_clust_er, global_norm_clust_er, player_clust_er, triad_clust_er = \
    hp.calculate_transitive_triad_intensity(G_null_er)

##### 8.1.4.1 Global Clustering
To start, the global internal clustering metric rises to $708.54$. This tells us the connectivity of the network has become richer and more interconnected. There now exists a higher proportion of high quality clusters pulling up the network's average. This happens because the graph is now highly connected and homogenous, it's not that there are localized clusters but now the whole network is made up of dense clusters.

In [ ]:
print("\n" + "="*80)
print(f"Global Team Transitive Triad Clustering (Mean): {global_internal_clust_er:.4f} pass units")
print("\n" + "="*80)

##### 8.1.4.2 Player-Level Clustering

In [ ]:
print("\n" + "="*80)
print("Player-Level Transitive Triad Summary")
print("="*80)
print(player_clust_er.to_string())

##### 8.1.4.3 Clustering Triads
That biggest give away that the Null has failed comes through inspecting the triads. The 2nd strongest triad comprises an $A \to B \to C$ connection between Sabrina D’Angelo, Emma Stina Blackstenius and Emily Ann Fox. That is the goalkeeper to the striker to the rightback. This is a near impossible passing triangle, let alone the strongest triad. This is visualised in the polygon plot as an absurd overlayering of pitch wide, non-local clusters.

In [ ]:
print("\n" + "="*80)
print("Top 10 Active Transitive Triads")
print("="*80)
top_triads_summary = []
for idx, t in enumerate(triad_clust_er[:10], start=1):
    p1, p2, p3 = t["players"]
    top_triads_summary.append({
        "Rank": idx,
        "Player 1 (Origin/Target)": p1,
        "Player 2 (Intermediate)": p2,
        "Player 3 (Target/Origin)": p3,
        "Total Capacity (Pass Units)": round(t["total_capacity"], 2)
    })

top_triads_df = pd.DataFrame(top_triads_summary).set_index("Rank")
print(top_triads_df.to_string())


##### 8.1.4.4 Clustering Polygon Plot

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 9.0), facecolor='#ffffff')

hp.draw_vertical_pitch(ax=ax)
hp.plot_passmap_on_pitch(G_null_er,ax=ax)
hp.plot_transitive_triads_updated(G=G_null_er, triads=triad_clust_er, top_n=4, ax=ax)

title=G_null_er.graph.get('team_name', 'Team')
ax.set_title(f"Top Transitive Triads (Passing Triangles)\n{title}", )
plt.tight_layout()
plt.show()

## 9. Markovian Null Model
Introduction to the section

---

Section Contents: 
- [9.1 Pass Dataset and Binning Mechanism](#91-pass-dataset-and-binning-mechanism)
    - [9.1.1 Construct League-Wide Pass Dataset](#911-construct-league-wide-pass-dataset)
    - [9.1.2 Position Counter](#912-position-counter)
        - [9.1.2.1 Original Positions](#9121-original-positions)
        - [9.1.2.2 Condensed Positions](#9122-condensed-positions)
    - [9.1.3 Pitch Grid](#913-pitch-grid)
        - [9.1.3.1 Plot Pass Bin on Grid](#9131-plot-pass-bin-on-grid)
- [9.2 Recipient Probability Distribution Model](#92-recipient-probability-distribution-model)
- [9.2.1 Build Spatial Probability Model](#921-build-spatial-probability-model)
    - [9.2.1.1 Query Individual Bin Probability Distributions](#9211-query-individual-bin-probability-distributions)
    - [9.2.1.2 Covert Sample to Player](#9212-covert-sample-to-player)
- [9.3 Pass Dataset Augmentation (Recipient Resample)](#93-pass-dataset-augmentation-recipient-resample)
- [9.4 Dianostics, Debugging and Sense Checking](#94-dianostics-debugging-and-sense-checking)
    - [9.4.1 Percentage of Matching Rewires](#941-percentage-of-matching-rewires)
    - [9.4.2 Change in Match Pass Share](#942-change-in-match-pass-share)
    - [9.4.3 Striker Matrix Bin Probabilties](#943-striker-matrix-bin-probabilties)
    - [9.4.4 Resampled Striker Receptions Pitch Plot](#944-resampled-striker-receptions-pitch-plot)
    - [9.4.5 League-Wide Striker Pass Percentages](#945-league-wide-striker-pass-percentages)
- [9.5 Resample Network](#95-resample-network)
    - [9.5.1 Resample PassMap](#951-resample-passmap)
- [9.6 Degree Analysis](#96-degree-analysis)
    - [9.6.1 Macro Degree Analysis](#961-macro-degree-analysis)
- [9.7 Network Validation](#97-network-validation)
    - [9.7.1 Null Validation](#971-null-validation)
    - [9.7.2 Null Validation Statistics](#972-null-validation-statistics)
    - [9.7.3 Metric Insight (Draft)](#973-metric-insight-draft)

---

### 9.1 Pass Dataset and Binning Mechanism
In order to build out our markovian processes we need a league-wide pass dataset. This gives us the raw passes which are the substrate for building networks. Deriving out 1st order generative processes directly on this raw data is better than working this the end product network because it is inherently spatially constrained and of course domain specifc as it is the true empirical representation of the game itself. 

#### 9.1.1 Construct League-Wide Pass Dataset
Aside from just extracting the valid pass data from the API we need a few extra steps. First we need to engineer the recipient position/role of each pass. The data comes with a recipient name but not their position. Connecting this piece of information is non-trivial because player positions/roles can change throughout the game. To simpilify this, we attach a players starting position as their fixed role/position id. 

In [ ]:
from statsbombpy import sb
from statsbombpy.api_client import NoAuthWarning

pass_match_records = []

for match in match_records:
    m_id = match['match_id']
    team_name = match["team"]
    top_11_players = match["top_11_players"]
    top_11_ids = {p['Player ID'] for p in top_11_players}
    player_id_to_name = {p['Player ID']: p['Player Name'] for p in top_11_players}

    events_df = sb.events(match_id=m_id)
    passes_df = hp.extract_top11_pass_events(events_df, team_name, top_11_ids)

    passes = hp.extract_passes_with_recipient_position(passes_df)
    pass_match_records.append(passes)

all_passes_df = pd.concat(pass_match_records, ignore_index=True)
print(f"\nTotal passes compiled across {len(matches_df)} matches: {len(all_passes_df):,}")

#### 9.1.2 Position Counter
Another decision made was to condense positions down to a standard "CB", "CM" format instead of the more granular format offered by StatsBomb which contains the side "LCB", "RCB". This is done to improve data density for positions but also during the Markovian training process, to ensure that tactic biases are not encoded. We want to model the position of a CB in general, the skew because left and right can be considered as team specific.

##### 9.1.2.1 Original Positions

In [ ]:
print(hp.get_recipient_position_counts(all_passes_df))

##### 9.1.2.2 Condensed Positions

In [ ]:
print(hp.get_recipient_position_counts(all_passes_df, mapped=True))

#### 9.1.3 Pitch Grid
Statbombs normalizes pitch dimensions to 120x80 and we scale this to a 100x100 format. However, the coordinates, despite being bounded, are still continuous numbers. To name the markovian process better and maximise data density, we use a binning approach where the pitch is split into (X,Y) bins. Our baseline for doing this is (10,10). This allows us to model probability distributions that happen in a specific bin, or between two bins. This binning process occurs in the pass dataset construciton. The start and end location fields which are formatted as (x_coord, y_coord) are binned to (bin_x, bin_y).

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 9.0))
hp.draw_pitch_with_grid(
    original_pitch_func=hp.draw_vertical_pitch,
    ax=ax,
    grid_size=(10, 10),
    show_bin_labels=True
)
plt.show()

##### 9.1.3.1 Plot Pass Bin on Grid

In [ ]:
hp.plot_position_reception_diagnostics(
    passes_df=all_passes_df,
    target_position=None,
    bin_filter=(0, 4),
    grid_pitch_func=hp.draw_pitch_with_grid,
    base_pitch_func=hp.draw_vertical_pitch,
    dot_alpha=0.25,
)


### 9.2 Recipient Probability Distribution Model
To establish a domain-consistent null model, we first implement a first-order Markovian process that reassigns pass recipients according to empirical, league-wide positional distributions. This method strictly preserves the spatial coordinates of each pass, preventing the generation of unrealistic pass trajectories. Because the underlying probability distribution is trained on full-season empirical data, the model naturally encodes realistic, spatially tuned passing behavior while maintaining the spatial integrity of the original dataset.

#### 9.2.1 Build Spatial Probability Model
For Null Model 1, we construct a first-order Markovian process that models the recipient's position given the spatial bin where a pass ends. To do this, we compute a frequency tensor of all passes received across every pitch bin, categorized by condensed recipient position. We apply Laplace smoothing ($\alpha = 1.0$) across all entries to eliminate zero-probability bins and maintain mathematical rigor. 

The function `build_prebinned_spatial_probability_model()` outputs both the raw `smoothed_counts` and normalized `prob_tensor` across all pitch bins, alongside coordinate mapping dictionaries (`pos_to_idx` and `idx_to_pos`).

In [ ]:
# Build spatial probability model
prob_tensor, smoothed_counts, pos_to_idx, idx_to_pos = \
    hp.build_prebinned_spatial_probability_model(
    passes_df=all_passes_df,
    bin_col="pass_end_bin",
    pos_col="recipient_position_11",
    grid_size=(10, 10),
    alpha=1.0,
    position_map=hp.POSITION_MAP_11
)

##### 9.2.1.1 Query Individual Bin Probability Distributions
`get_position_distribution_for_bin()` queries a specific grid cell $(r, c)$ to return the conditional probability distribution of likely recipient positions.

In [ ]:
bin_distribution = \
    hp.get_position_distribution_for_bin(
    r=5, c=5, #(x,y)
    prob_tensor=prob_tensor, 
    idx_to_pos=idx_to_pos,
    as_df=True
)
print(bin_distribution)

##### 9.2.1.2 Covert Sample to Player
Demonstating the ability to sample from the distribution

In [ ]:
# Sample a single position string
sampled_position = np.random.choice(
    bin_distribution['Position'],
    p=bin_distribution['Probability']
)

sampled_player, simpled_id = hp.get_random_player_by_short_position(
    target_record['top_11_players'], sampled_position)

print(f"Sampled Position: {sampled_position}")
print(f"Sampled ID: {simpled_id}")
print(f"Sampled Sampled: {sampled_player}")

### 9.3 Pass Dataset Augmentation (Recipient Resample)
To generate a synthetic variant of a target match, we construct a resampled pass dataset. The spatial attributes of each pass remain identical to the empirical event, but the recipient is "rewired" by sampling from the probability distribution of the terminal pass bin. A pass can be rewired back to its original empirical recipient, which frequently occurs in pitch locations where that player is the spatially dominant receiver.

This resampling procedure generates an entirely new edge list ($A \to B$ interactions) from which to construct our network. As the origin events and player rosters remain unchanged, the node set remains constant while the pairwise edge connections vary.

In [ ]:
MATCH = target_record["match_id"]
TEAM = target_record["team"]

sample_match = all_passes_df[
    (all_passes_df['match_id'] == MATCH) &
    (all_passes_df['team'] == TEAM)
    ].copy()

resampled_match_df = hp.resample_pass_recipients(
    sample_match_df=sample_match,
    prob_tensor=prob_tensor,
    idx_to_pos=idx_to_pos,
    top_11_players=target_record['top_11_players'],
    seed=100
)

# View the original passer alongside the newly resampled recipient
print(resampled_match_df[
    ['player', 'player_id', 
     'pass_recipient', 'pass_recipient_id', 
     'resampled_recipient', 'resampled_recipient_id', 
     'recipient_position_11', 'resampled_recipient_position']])

### 9.4 Dianostics, Debugging and Sense Checking
This section represents some ad-hoc diagnostics to make sense of null outputs

##### 9.4.1 Percentage of Matching Rewires
Around 15% of the pass remaps resulting in the same player. This is fine and maybe a perfect model would be a bit higher but at a minimum if varifies that we are not generating the same input data.

In [ ]:
match_pct = (
    resampled_match_df['pass_recipient'] == \
        resampled_match_df['resampled_recipient']).mean() * 100

print(f"Exact match (ignoring NaNs): {match_pct:.2f}%")

##### 9.4.2 Change in Match Pass Share
One of the key things to look at is how the rewiring changes total pass recipient values:

**Analysis:**
Foord and Mead, the wingers, jump up in their share. However, in the original match, they only play 65 minutes each so this resampling is to be expected as it has no inference of temporal aspects. The resample essentially treats them as if they played 90 minutes hence the increase in frequency. This is a limitation of the project but a nessecary step for simplicity. 

Sabrina D’Angelo, the goalkeeper, maintains a similar amount of passes at going from 19 to 13. This is good becuase during the ER null configuation, the goalkeeper turned into a key playmaker. This demonstates that the null understands role constraints. 

Interestingly, Carlotte Wubben-Moy passes fall from 120, the highest on the pitch, to 62, inline with her CB partner, Leah Williamson who falls from 99 to 67. At this level, they are still key hubs on the pitch receiving 8.76 and 9.37 of the passes but are less "hub" like than before. 

The biggest change comes from Emma Stina Blackstenius, the striker, who in the real match only recived 10 passes from a total 840(720) passes in the match. This is an unusually low tally but this is because she is a unique player who fundementally does not get involved in possesion. She plays as terminal striker and is known for the hard off the ball work without touching the ball. However, in the rewire, she is allocated 104 passes giving her $14.55$ of the receptions. 

There are two things to consider here. This was was an unusual game. Arsenal won 5-0 against a much weaker team, therefore, alot of the passes were taken high up the pitch in the oppositions half. In these high areas, the model allocates a strong, though often not the top, percentage to the striker. Due to the volumne of the high pass, a large percentage of these passes were allocated to her. Additionally, the striker allocation is based on the league average. In modern football, a lot of top level strikers like to drop deep and be involved in possesion. Blackstenius is an outlier in this sense. 

In [ ]:
# Calculate frequency counts and percentage distributions
orig_counts = resampled_match_df["pass_recipient"].value_counts()
orig_pct = resampled_match_df["pass_recipient"].value_counts(normalize=True) * 100

resamp_counts = resampled_match_df["resampled_recipient"].value_counts()
resamp_pct = resampled_match_df["resampled_recipient"].value_counts(normalize=True) * 100

# Combine metrics into a single comparison DataFrame
recipient_distribution_df = pd.DataFrame({
    # "Position": resampled_match_df['position'],
    "Original Passes Received (Count)": orig_counts,
    "Original Share (%)": orig_pct.round(2),
    "Resampled Passes Received (Count)": resamp_counts,
    "Resampled Share (%)": resamp_pct.round(2)
}).fillna(0) 

# Sort by Original Pass Volume descending
recipient_distribution_df = recipient_distribution_df.sort_values(
    by="Original Passes Received (Count)", ascending=False
)

# Display Summary Table
print(recipient_distribution_df.to_string())


##### 9.4.3 Striker Matrix Bin Probabilties

In [ ]:
import numpy as np

# Identify the tensor slice index for the Striker position
striker_code = "ST"  # Update if your condensed position uses another code (e.g., 'CF')
striker_idx = pos_to_idx[striker_code]

# Slice the 10x10 matrix of P(Striker | Grid Cell)
# Slicing prob_tensor[:, :, striker_idx] yields shape (n_rows, n_cols)
striker_prob_matrix = prob_tensor[:, :, striker_idx]

print(f"Striker Probability Matrix Shape: {striker_prob_matrix.shape}")
print("\nStriker Spatial Probability Matrix (P(ST | Bin)):\n")
print(np.round(striker_prob_matrix, 4))

##### 9.4.4 Resampled Striker Receptions Pitch Plot
This shows us the areas where the striker was resample receptions. Note that a lot of the touchers are deep in the final third and almost all in the oppositions half. When taking into account the sheer volume of passes in this match, this plot looks accurate for a striker. 

In [ ]:
hp.plot_position_reception_diagnostics(
    passes_df=resampled_match_df,
    target_position="ST",
    grid_pitch_func=hp.draw_pitch_with_grid,
    base_pitch_func=hp.draw_vertical_pitch,
    dot_alpha=0.25,
    sample="resample"
)

##### 9.4.5 League-Wide Striker Pass Percentages
To sense check the data, the league-wide striker pass percentage is computed and it comes out to 5.39%. However, as noted, our sample match was not a normal game. Rarely to teams dominate and play to high up the pitch, therefore, the 14% allocated to a striker look valid as a result of extreme variance. 

In [ ]:
COMPETITION_ID = 37
SEASON_ID = 281
matches = sb.matches(competition_id=COMPETITION_ID, season_id=SEASON_ID)

_p = all_passes_df.copy()

hp.calculate_striker_pass_percentage(
    match_ids=matches["match_id"].unique(), 
    striker_positions=["Center Forward", "Left Center Forward", "Right Center Forward"], 
    passes=_p
)

### 9.5 Resample Network
Now that we have a new set of A to B pass events we can build a network using the same function

In [ ]:
G_rec_null = build_passmap_network(
    target_record,
    resampled_match_df,
    null=True)

G_rec_null.graph['team_name'] = 'Arsenal WFC (Rec Null #1)'


#### 9.5.1 Resample PassMap

In [ ]:
# Create a figure with 1 row and 2 columns
fig, ax = plt.subplots(1, 2, figsize=(13.0, 9.0), facecolor='#ffffff')

# Plot
hp.plot_passmap_frameless(G_highest, ax=ax[0]) #left 
hp.plot_passmap_frameless(G_rec_null, ax=ax[1])

# Display both side by side
plt.tight_layout()
plt.show()

### 9.6 Degree Analysis
The degree analysis for this single null reveals an interesting set of stories. 

ER null collapsed $G(N, p)$ down to 514.99 from 5681.90. THe recipient rewire preserves the metric to $\text{Var}(s_{\text{tot}}) = 2955.90$. Because we preserve the exact passes ($s_i^{\text{out}}$) we recover  over half of the empirical volume variance ($\text{Var}(s_{\text{tot}})$. This proves that passer initiative (who starts the pass and how often) accounts for roughly $50\%$ of team volume centralization, while receiver availability drives the remaining $50\%$. The increase of **Unweighted Connection Density** to ($\langle k \rangle = 17.6364$) means the rewired network becomes even denser ($\approx 88\%$ of all possible directed channels are active). This could be variance, or it could be because low freq channels which were ignores tacticly are bound back in by the league average behaviour. This also explains why the network become more homogenizaed abstracted links like Stina Blackstenius were wrapped back in. 

In [ ]:
# Analyze graph using the integrated function
player_df_rec_null, macro_stats_rec_null, top_hubs_df_rec_null = \
    hp.analyze_degree_and_heterogeneity(G_rec_null, top_n_hubs=5)

#### 9.6.1 Macro Degree Analysis

| Metric | Real Empirical Network (Arsenal WFC) | Erdős–Rényi ($G(N,p)$) | Tier 1: Recipient Rewired Null | Diagnostic Trend |
| :--- | :---: | :---: | :---: | :--- |
| **Mean Unweighted Degree ($\langle k \rangle$)** | 16.1818 | 15.4545 | 17.6364 | **Topological Oversaturation:** Unconstrained spatial recipient draws activate additional minor passing channels. |
| **Coefficient of Variation ($CV_k$)** | 0.1724 | 0.1305 | 0.1115 | **Highest Homogenization:** Shuffling recipients via league spatial distributions flattens unweighted degree variance even more than ER. |
| **Normalized Second Moment ($\frac{\langle k^2 \rangle}{\langle k \rangle}$)** | 16.6629 | 15.7176 | 17.8557 | **Structural Preservation:** Higher than ER because fixed empirical pass origins ($(x_1, y_1)$) anchor structural density near real values. |
| **Node Volume Variance ($\text{Var}(s_{\text{tot}})$)** | 5681.90 | 514.99 | 2955.90 | **Partial Volume Recovery:** Recovers $\sim 52\%$ of real volume variance because empirical passer volume is preserved. |

In [ ]:
macro_stats_rec_null

### 9.7 Network Validation
We need to validate the null networks we create to determine that we are suitable for analysis. So far we have some up with the following conditions: Hetrogenity; Small-word (according to the lit); Goalkeeper not hubs; Null appropriately different from the input network.Note that the goal is not to have a comprehsive finalise evaluation framework but instead to start the chain of one given the liturature does not have one. 

I think these results are really great. We are destroying the tactical naunces of the network and replacing it was a generic "football" network. We are retaining football composition by reusing the actual real passes and maining true relationship by training the prob distribution on the real binned data. What is happening is that the tactic experiences of the underling network are being washed out but the topological contrains remain. It gives us a chance to really highlight what our given network did that was interesting

##### Metrics to Look at:

Coefficient of Variation ($CV_k$) and the Normalized Second Moment ($\frac{\langle k^2 \rangle}{\langle k \rangle}$) are unequivocally the best metrics for testing and evaluating degree heterogeneity. 

Coefficient of Variation ($CV_k$): $CV_k$ is scale-invariant. It normalizes the standard deviation against the mean degree, allowing you to compare relative heterogeneity directly across different networks, match samples, or null generated ensembles regardless of total edge volume.  $CV_k \approx 0$: Complete homogeneity (a regular grid where every player connects to the exact same number of partners). $CV_k \approx 0.15\text{--}0.25$: Typical football passing network topology (mild heterogeneity, reflecting a dense graph with minor positional channel variation).  $CV_k > 1.0$: Strong heterogeneity (a heavy-tailed, scale-free network dominated by extreme hubs)


Normalized Second Moment / Heterogeneity Ratio ($\frac{\langle k^2 \rangle}{\langle k \rangle}$):  In network science (particularly when modeling spreading dynamics or structural robustness), comparing the second moment $\langle k^2 \rangle$ directly to the mean $\langle k \rangle$ determines how much hub creation distorts the graph topology relative to a uniform random graph.  

$\frac{\langle k^2 \rangle}{\langle k \rangle} \approx \langle k \rangle$: The network is homogeneously connected with minimal degree fluctuation (standard for an $11 \times 11$ football passmap). $\frac{\langle k^2 \rangle}{\langle k \rangle} \gg \langle k \rangle$: The network features extreme degree variance and distinct hub nodes.

Team Node Volume Variance Var(s_tot)

#### 9.7.1 Null Validation
To validate that this null process is appropriate we need to execute the process many times to build up a range of output metrics. Runs 500 times with different seeds. Each iteration destorys the A to B and resamples the B based on the league avergae behaviour. Above we analysed 1 null formulation but this just represents a single formulation from the distribution. We need to compute a range of nulls to understand if the Null model has the capacity to veere into unrealistically bounds.

In [ ]:
N_ITERATIONS = 500

# Target Metric Tracking
mean_k_list = []
cv_k_list = []
second_moment_ratio_list = []
var_stot_list = []
keeper_volume = []
adj_correl = []
top_retention = []

for seed in range(N_ITERATIONS):

    resampled_match_df = hp.resample_pass_recipients(
        sample_match_df=sample_match,
        prob_tensor=prob_tensor,
        idx_to_pos=idx_to_pos,
        top_11_players=target_record['top_11_players'],
        seed=seed
    )
    
    # Build Null Network
    G_null = build_passmap_network(
        target_record,
        resampled_match_df,
        null=True
    )

    player_df_rec_null, macro_stats_rec_null, top_hubs_df_rec_null = \
    hp.analyze_degree_and_heterogeneity(G_null, top_n_hubs=5)

    _r_val = hp.compute_adjacency_correlation(G_highest, G_null)

    # Extract and store the target metrics from the dictionary
    mean_k_list.append(macro_stats_rec_null['Mean Unweighted Degree <k>'])
    cv_k_list.append(macro_stats_rec_null['Coefficient of Variation (CV_k)'])
    second_moment_ratio_list.append(macro_stats_rec_null['Normalized Second Moment (<k^2> / <k>)'])
    var_stot_list.append(macro_stats_rec_null['Team Node Volume Variance Var(s_tot)'])
    keeper_volume.append(player_df_rec_null['Total Volume (s_tot)']['Sabrina D’Angelo'])
    adj_correl.append(_r_val)


# ompile statistical summary table
metrics_data = {
    'Mean (<k>)': mean_k_list,
    'CV_k': cv_k_list,
    'Second Moment Ratio (<k^2>/<k>)': second_moment_ratio_list,
    'Node Vol Variance Var(s_tot)': var_stot_list,
    'Keeper Total Volume': keeper_volume,
    'Adj Correlation': adj_correl,
}

summary_df = pd.DataFrame({
    'Metric': list(metrics_data.keys()),
    'Null Mean': [np.mean(vals) for vals in metrics_data.values()],
    'Null Std': [np.std(vals) for vals in metrics_data.values()],
    'Min': [np.min(vals) for vals in metrics_data.values()],
    'Max': [np.max(vals) for vals in metrics_data.values()],
    '95% CI Lower': [np.percentile(vals, 2.5) for vals in metrics_data.values()],
    '95% CI Upper': [np.percentile(vals, 97.5) for vals in metrics_data.values()]
})

#### 9.7.2 Null Validation Statistics

In [ ]:
print("============================================================")
print(f"Tier 1 Null Model Resampling Summary (N={N_ITERATIONS} Iterations)")
print("============================================================")
print(summary_df.to_string(index=False))

#### 9.7.3 Metric Insight (Draft)

**Mean Unweighted Degree ($\langle k \rangle$).**

The impirical was 16.18 and the mean was 17.74 from a possible max of $20$ in an 11 plaeyr nework. An average degree of $17.74$ represents an active connection density of $\approx 88.7\%$. We expect a dense netowrk, particualrly with this many passes but it is good see tha tthere are no max netowrks, nor a skew towards. The fact that the empirical network is lower then the mean just tell us that there are really tactic decison taking place to leaves specific passing channels unused, whereas random spatial reallocation populates peripheral channels with 1–2 passes, creating a slightly oversaturated graph.

**Coefficient of Variation ($CV_k$)**: 

Null Ensemble: Mean = 0.1201, Range = [0.0665, 0.1858], 95% CI = [0.0759, 0.1707]. Compare to Empirical Arsenal Value: 0.1724. None of the generated networks hit $CV_k \ge 1.0$ (which would indicate an unrealistic scale-free star network). However, the lower min (0.0665) approaches near-perfect lattice homogeneity. The empirical value of $0.1724$ sits above the 95% CI upper bound ($0.1707$). This is a critical finding: real passing networks possess significantly higher structural heterogeneity than spatial occupancy alone dictates. Reallocating recipients via league-average distributions washes out tactical role separation, forcing nodes toward uniform connectivity profiles. The fact that arsneal didnot homogenize with this many passes is the insight. These lower bound networks are just possibilities at thispass level, not mistakes.

**Normalized Second Moment ($\frac{\langle k^2 \rangle}{\langle k \rangle}$):** 

Null Ensemble: Mean = 18.00, Range = [17.09, 18.84], 95% CI = [17.31, 18.66]. Across all 100 iterations, $\frac{\langle k^2 \rangle}{\langle k \rangle} \approx \langle k \rangle$ holds true. This confirms that Tier 1 never accidentally produces a distorted "star network" hub (e.g., a goalkeeper absorbing all connections). The ratio tracks closely with the mean degree, proving that the generative engine respects pitch boundary conditions and avoids scale-free structural anomalies.

When we talk about hubs in football is subtle definition. we don't expect to see superhubs whereby the network cannot function without them. 

**Node Volume Variance ($\text{Var}(s_{\text{tot}})$):** 

Empirical Arsenal Value: 5681.90, Null Ensemble: Mean = 3126.62, Range = [2620.63, 3705.54], 95% CI = [2708.09, 3572.05]. Unlike Erdős–Rényi $G(N, p)$—which collapses volume variance down to $514.99$—Tier 1 recovers over $55\%$ of the empirical volume variance ($3126.62 / 5681.90$).  Because Tier 1 holds each passer's outgoing volume ($s_i^{\text{out}}$) fixed, it preserves the primary volume anchors (central defenders and double pivots). The empirical match’s value ($5681.90$) far exceeds the null's max ($3705.54$), proving that while passer initiative drives $55\%$ of workload inequality, targeted receiver choice accounts for the remaining $45\%$

## 10. Null Baselined Analysis

**Global Metrics, run 500 times and collect:**
- Global Path
- Global Clustering

**Meso metrics, run 500 times and collect perms:**
- Triad. compute perms and accrue triad strength over 500

**Player-level, runn 500 and collect by position:**
- Bet Centrality
- Path Player
- Clustering

# Extra

Stina Blackstenius is a distinct off the ball striker and this is well documents (can i reind references)

In [ ]:
#Specific Match Details
for index, row in matches_df.iterrows():
    if row['match_id'] == 3913160:
        print(row)

- #24 C. Lacasse on for #19 C. Foord 63'
- #15 K. McCabe on for #9 B. Mead 63'
- #26 L. Wienroither on for #2 E. Fox 81'
- #62 K. Reid on for #6 L. Williamson 87'